### Import Libraries

In [0]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pyspark.sql.functions import col,trim,lower,concat

### Run Utility Notebooks

In [0]:
%run ../utils/params

In [0]:
 %run ../utils/utilsMethods

In [0]:
%run ../utils/adlsConfig

### Read Data From adhocRun Table

In [0]:
adhocRunTableDf = spark.sql(f"select * from {adhocRunTable} where lower(trim(adhocRunFlag))='y'")
adhocRunTableDf = adhocRunTableDf.withColumn("sourceInputObject", concat("sourceSystem",lit("~"),"sourceObjectName"))
sourceList = adhocRunTableDf.select("sourceInputObject").rdd.map(lambda x : x[0]).collect()

### Run rawToCuration in Parallel

In [0]:
try:
    with ThreadPoolExecutor(8) as executor:
        results = executor.map(adhocRawToCuration, sourceList)
except Exception as e:
    executor.shutdown(wait=False)
    raise e


In [0]:
dbutils.notebook.exit("Adhoc Raw To Curation ran successfully")